# RNN（循環神經網路，1986）

- **核心原理**：隱藏狀態在時間步之間傳遞
- **變體**：Bi-RNN（雙向）、多層（stacked）RNN；下一課的 LSTM／GRU
- **推薦資料集**：名字分類 —— 18 種語言、約兩萬個姓氏（PyTorch 官方教學「NLP From Scratch: Classifying Names with a Character-Level RNN」附的資料）
- **論文**：Rumelhart, Hinton, Williams (1986), *Learning representations by back-propagating errors*；Elman (1990), *Finding Structure in Time*

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 1 課（MLP）、第 5 課（LeNet）的差別：

| | MLP／LeNet | 本章 RNN |
| --- | --- | --- |
| 輸入 | 固定大小（784 個像素、32×32 影像） | **長度不一的序列**（名字 2～20 個字母） |
| 參數共用 | LeNet：同一個濾波器用在每個**位置** | 同一組 $W_x, W_h$ 用在每個**時間步** |
| 手寫的層 | `Linear`、`Conv2d`… | 沿用第 1 課的層，**新增** `Embedding`（查表）、`RNN`（隱藏狀態 + 時間上的反向傳播 BPTT） |
| 批次 | 每筆一樣大，直接疊成矩陣 | 補到一樣長（padding）+ **mask** 記住哪些位置是真的字母 |
| 新問題 | — | **梯度消失／爆炸**（連乘 $T$ 次）、梯度裁剪 |

一句話：**RNN 是「同一個小 MLP，沿著序列一步一步重複使用，並把上一步的輸出（隱藏狀態）當成這一步的額外輸入」。**

這一章的重點：**字元怎麼變成向量（one-hot 與 embedding）、隱藏狀態的遞迴、padding 與 mask、BPTT（時間上的反向傳播）、為什麼梯度會消失或爆炸、梯度裁剪、讀的方向與雙向 RNN，以及一個誠實的問題：在這份資料上，RNN 比得過「字母 n-gram + 邏輯回歸」嗎？**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：每筆是一個姓氏（例如 `Nakamura`）與它的語言（`Japanese`）。字母先轉成 ASCII（去掉重音：`Ślusàrski` → `Slusarski`），每個字母對應一個整數編號（0 留給 padding）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | shape／本章的值 |
| --- | --- | --- |
| $n$ | 一個 batch 的名字數 | 64 |
| $T$ | batch 裡最長的名字長度（其他補 0） | 2～20 |
| $L_i$ | 第 $i$ 個名字的真正長度 | — |
| $V$ | 字母表大小（含 padding 的 0） | 59 |
| $E$ | 字母向量（embedding）的維度 | 32 |
| $H$ | 隱藏狀態的維度 | 128 |
| $K$ | 類別數（語言） | 18 |
| $x_t$ | 第 $t$ 個字母的向量 | $(n, E)$ |
| $h_t$ | 讀完第 $t$ 個字母後的隱藏狀態；$h_0 = 0$ | $(n, H)$ |
| $W_x$、$W_h$、$b$ | RNN 的參數：輸入→隱藏、隱藏→隱藏、偏差 | $(E, H)$、$(H, H)$、$(H,)$ |
| $m_t$ | mask：第 $t$ 個位置是不是真的字母（1／0） | $(n,)$ |
| $z$ | 輸出層的 logits $= h_{L} W_o + b_o$ | $(n, K)$ |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 課 | MLP＋反向傳播（`03-early-nn/01-mlp-backprop`） | 每一層有 `forward`／`backward`；`Linear` 的梯度 $\partial L/\partial W = X^\top \delta$、傳給前一層 $\delta W^\top$；softmax＋交叉熵的梯度 $(P - Y)/n$；`tanh` 的導數 $1 - \tanh^2$；梯度檢查用中央差分，相對誤差應 < 1e-6。 |
| 第 5 課 | LeNet-5 | 權重共用：同一個濾波器滑過每個位置，梯度是**每個位置的梯度加總**。RNN 在時間上做同一件事。 |
| 第一部分第 6 章 | 樸素貝氏（SMS 垃圾簡訊） | 文字用「詞袋」（bag of words）表示：只數出現次數、不管順序；字母 n-gram 是同樣的想法，只是單位換成連續 n 個字母。 |
| 第一部分第 20、21 章 | HMM、CRF | 序列模型：前一步的狀態影響這一步。HMM 的隱藏狀態是**離散**的（$K$ 種），RNN 的是**連續**向量（$H$ 維）。forward 演算法也是「一步一步往後傳」。 |
| 第一部分第 20 章 TODO 11 | 監督式 HMM 的數數 | `np.add.at(B, (z, x), 1)`：同一格出現多次也會累加（`B[z, x] += 1` 不會）。本章 Embedding 的反向傳播用同一招。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 多變數鏈鎖律**：$L$ 經過好幾條路徑依賴 $h$ 時，梯度是**每條路徑的梯度加總**。例：$L = f(a) + g(a)$、$a = 2h$ → $\partial L/\partial h = 2f'(a) + 2g'(a)$。RNN 的 $h_t$ 有兩條路影響 loss：直接（如果它是最後一步）以及透過 $h_{t+1}$。

**2. tanh 的導數**：$\frac{d}{da}\tanh(a) = 1 - \tanh^2(a)$。例：$\tanh(0) = 0$ → 導數 1；$\tanh(2) = 0.964$ → 導數 $0.071$（**飽和**：輸入大時梯度很小）。

**3. 連乘的大小**：$0.9^{20} = 0.12$、$0.5^{20} \approx 10^{-6}$、$1.5^{20} \approx 3{,}300$。$T$ 步的 BPTT 會把「每一步的縮放倍數」連乘起來 → 不是消失就是爆炸。

**4. 矩陣的譜半徑**：$\rho(W) = \max|\lambda_i|$（特徵值的最大絕對值）。$v \mapsto vW$ 重複很多次，長度大約乘上 $\rho^t$。正交矩陣 $\rho = 1$、而且不改變長度。

**5. 向量的範數**：$\lVert g\rVert = \sqrt{\sum g_i^2}$；**全域範數**是把所有參數的梯度串成一條長向量再算。

**6. 交叉熵與 macro-F1**（第一部分）：accuracy 被大類別主導；**macro-F1** = 每一類各算 F1 再平均（每類同樣權重）→ 小類別也算數。

#### 程式

**7. one-hot 與查表**：字母編號 $c$ 的 one-hot 向量乘上矩陣 $W_{\text{emb}}$ $(V, E)$ = 取出第 $c$ 列：`onehot(c) @ W == W[c]`。一次查一整個 batch：`W[X]`，`X` 是 $(n, T)$ 的整數 → 結果 $(n, T, E)$。
**8. `np.add.at(A, idx, vals)`**：`A[idx] += vals`，但同一個 index 出現多次時**每次都加**。（`A[idx] += vals` 只會加最後一次。）
**9. mask 的寫法**：`m = mask[:, t, None]`（$(n, 1)$，可以跟 $(n, H)$ 廣播），`h = m * h_new + (1 - m) * h_old` → mask = 0 的名字保持舊的 $h$。
**10. 字串**：`unicodedata.normalize("NFD", s)` 把 `é` 拆成 `e` + 重音符號；`unicodedata.category(c) == "Mn"` 是重音符號。`s[::-1]` 反轉字串。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。每格附了一兩個引導問題，回答完就可以把問題刪掉。

`load_names()` 的回傳格式：`{語言: [名字, ...]}`（原始資料，**包含重複**）。下一格把它整理成一張表。

In [ ]:
import string
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_names

names_raw = load_names()
if isinstance(names_raw, dict):
    raw = pd.DataFrame([(nm, lang) for lang, ns in names_raw.items() for nm in ns], columns=["name", "language"])
else:  # 也接受 DataFrame（欄位 name、language）
    raw = pd.DataFrame(names_raw)[["name", "language"]]
LANGS = sorted(raw["language"].unique())
K = len(LANGS)
print("原始筆數：", len(raw), "；語言數：", K)

per_lang = raw.groupby("language")["name"].agg(rows="size", unique="nunique")
per_lang["duplicate rows"] = per_lang["rows"] - per_lang["unique"]
display(per_lang.sort_values("rows", ascending=False).T)

In [ ]:
# 觀察：
#   - 哪個語言的「重複」最多？去掉重複之後，它還剩幾個名字？
#   - 有多少個名字同時出現在兩種以上的語言？舉幾個例子 —— 這些名字「正確答案」是什麼？
df = raw.drop_duplicates().reset_index(drop=True)
multi = df.groupby("name")["language"].agg(lambda s: tuple(sorted(s)))
multi = multi[multi.map(len) > 1]
print(f"去重後：{len(df)} 筆（{df['name'].nunique()} 個不同的名字）；出現在 ≥ 2 種語言的名字：{len(multi)} 個")
print("例子：", multi.head(12).to_dict())
pair_counts = Counter(p for langs in multi for p in [(a, b) for i, a in enumerate(langs) for b in langs[i + 1 :]])
print("最常重疊的語言對：", pair_counts.most_common(8))

In [ ]:
# 字母表：轉 ASCII、去掉重音
# 觀察：
#   - 原始資料裡有哪些非 ASCII 字元？轉成 ASCII 之後，有沒有名字因此變得跟另一個名字一樣？
#   - 名字長度的分布：最短、最長、中位數？不同語言的長度差多少（看箱形圖）？
ALLOWED = string.ascii_letters + " .,;'-"
CHAR_TO_ID = {c: i + 1 for i, c in enumerate(ALLOWED)}  # 0 留給 padding
V = len(ALLOWED) + 1


def to_ascii(s: str) -> str:
    """去掉重音、只留 ALLOWED 裡的字元。例：'Ślusàrski' → 'Slusarski'。"""
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn" and c in ALLOWED)


non_ascii = sorted({c for nm in df["name"] for c in nm if c not in string.ascii_letters})
print("非 ASCII 英文字母的字元：", non_ascii)
df["ascii"] = df["name"].map(to_ascii)
df["y"] = df["language"].map({lang: k for k, lang in enumerate(LANGS)})
df["length"] = df["ascii"].str.len()
print("例：", {nm: to_ascii(nm) for nm in ["Ślusàrski", "Müller", "Nuñez", "O'Brien", "Abl-Al-Hamid"]})
print(f"字母表大小 V = {V}（{len(ALLOWED)} 個字元 + padding）；轉 ASCII 後撞名（同一個 ASCII 字串、不同原名）：{df.groupby('ascii')['name'].nunique().gt(1).sum()} 個")

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].hist(df["length"], bins=np.arange(1, 22) - 0.5)
axes[0].set_xlabel("name length (characters)")
order = df.groupby("language")["length"].median().sort_values().index
sns.boxplot(data=df, x="language", y="length", order=order, ax=axes[1])
axes[1].tick_params(axis="x", rotation=60)
plt.show()
print(df["length"].describe().round(2).to_dict())

In [ ]:
# 字尾：每種語言最常見的「最後 2 個字母」與「最後 3 個字母」
# 觀察：
#   - 哪些語言有非常明顯的字尾（例如 -ov、-ez）？哪些語言沒有？
#   - 字首呢？（例如 Mc-、O'-）你覺得名字的哪一端資訊比較多？
def top_affix(series: pd.Series, k: int, end: bool = True, n_top: int = 4) -> str:
    aff = series.str.lower().map(lambda s: s[-k:] if end else s[:k])
    vc = aff.value_counts(normalize=True).head(n_top)
    return ", ".join(f"{a}({p:.0%})" for a, p in vc.items())


affix_tbl = pd.DataFrame({
    lang: {"n": len(g), "suffix-2": top_affix(g["ascii"], 2), "suffix-3": top_affix(g["ascii"], 3), "prefix-2": top_affix(g["ascii"], 2, end=False)}
    for lang, g in df.groupby("language")
}).T.sort_values("n", ascending=False)
display(affix_tbl)

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 類別不平衡：
#     (a) 最大、最小的語言各有幾個名字（去重後）？差幾倍？「永遠猜最大類」的 accuracy 大約多少？
#     (b) 為什麼本章除了 accuracy，還要看 macro-F1（前置複習 6）？如果模型「永遠不猜 Vietnamese」，accuracy 和 macro-F1 各會受多大影響？
# 答：
#
# (2) 重複與歧義：
#     (a) 原始資料哪個語言的重複最嚴重？如果不去重就隨機切 train／test，同一個名字會同時出現在兩邊 —— 測試成績會被高估還是低估？（第 2 節會實測）
#     (b) 同時屬於兩種語言的名字（例如出現在 English 和 Scottish）：任何模型在它們身上的 accuracy 上限是多少？這叫什麼（第一部分：不可約誤差／Bayes error）？
# 答：
#
# (3) 順序重要嗎？
#     (a) 從字尾表找三個「看最後 2～3 個字母就幾乎能確定」的語言。
#     (b) 「Ivanov」和「Vonavi」有一樣的字母、不同的順序。詞袋（只數字母出現次數）分得出來嗎？字母 n-gram（連續 2～3 個字母）呢？RNN 呢？
# 答：

### 資料切分：依「名字」分組的分層切分

去重之後，有些名字仍然出現在不止一種語言（同一個字串、不同標籤）。如果同一個字串一筆在 train、一筆在 test，模型等於在 test 看到「背過的字串」。所以用 `StratifiedGroupKFold`：

- **group** = ASCII 名字（同一個字串的所有列一定在同一邊）；
- **stratified** = 每一邊的語言比例大致相同（小語言才不會整個不見）。

切成 train 約 67%、validation 約 17%、test 約 17%。調參全程只用 train／validation，test 到第 6 節才看一次。

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

rest_idx, test_idx = next(StratifiedGroupKFold(6, shuffle=True, random_state=0).split(df, df["y"], groups=df["ascii"]))
df_rest = df.iloc[rest_idx]
tr_idx, va_idx = next(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(df_rest, df_rest["y"], groups=df_rest["ascii"]))
df_tr, df_va, df_te = df_rest.iloc[tr_idx].reset_index(drop=True), df_rest.iloc[va_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)
y_tr, y_va, y_te = df_tr["y"].to_numpy(), df_va["y"].to_numpy(), df_te["y"].to_numpy()
print("train / val / test：", len(df_tr), len(df_va), len(df_te))
print("同一個 ASCII 名字跨 split？", bool(set(df_tr["ascii"]) & set(df_te["ascii"])), bool(set(df_tr["ascii"]) & set(df_va["ascii"])))
split_counts = pd.DataFrame({"train": df_tr["language"].value_counts(), "val": df_va["language"].value_counts(), "test": df_te["language"].value_counts()}).loc[LANGS]
display(split_counts.T)

---
## 2. Baseline

| Baseline | 做法 | 看什麼 |
| --- | --- | --- |
| Dummy（prior） | 永遠輸出訓練集的類別比例 | 最低標準 |
| 字母 1-gram + 邏輯回歸 | 只數每個字母出現幾次（詞袋，**沒有順序**） | 順序完全不用，能做到多少？ |
| 字母 1～3-gram + 邏輯回歸 | 也數連續 2、3 個字母（**局部順序**） | 「局部順序」值多少？這是 RNN 要打敗的對手 |

`CountVectorizer(analyzer="char_wb", ngram_range=(1, 3))`：把 `"ivanov"` 變成 `" i", "iv", "va", ..., "ov ", " iv", ...`（`char_wb` 會在字的前後加空白，所以字首、字尾的 n-gram 是分開的特徵）。

**評估指標**（validation，之後一路往下加到 `val_table`）：accuracy、macro-F1、log loss（交叉熵）。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.pipeline import make_pipeline


def val_scores(prob: np.ndarray, y: np.ndarray = y_va) -> dict[str, float]:
    """prob (n, K) 是預測機率。回傳 {"accuracy", "macro_f1", "log_loss"}。"""
    pred = prob.argmax(axis=1)
    return {
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, average="macro", labels=np.arange(K), zero_division=0), 4),
        "log_loss": round(log_loss(y, np.clip(prob, 1e-12, 1), labels=np.arange(K)), 4),
    }


val_table = pd.DataFrame(columns=["accuracy", "macro_f1", "log_loss"])
dummy = DummyClassifier(strategy="prior").fit(df_tr[["ascii"]], y_tr)
val_table.loc["Dummy prior"] = val_scores(dummy.predict_proba(df_va[["ascii"]]))
ngram_models = {}
for ng in [(1, 1), (1, 2), (1, 3)]:
    m = make_pipeline(CountVectorizer(analyzer="char_wb", ngram_range=ng, lowercase=True), LogisticRegression(max_iter=3000))
    ngram_models[ng] = m.fit(df_tr["ascii"], y_tr)
    val_table.loc[f"char {ng[0]}-{ng[1]}-gram + LogReg"] = val_scores(m.predict_proba(df_va["ascii"]))
display(val_table)

In [ ]:
# 不去重、隨機切分（原始教學的做法）會怎樣？用同樣的 1～3-gram + LogReg
# 觀察：
#   - 不去重時的 accuracy 比上面高還是低？哪一個語言差最多（提示：看重複最多的那個）？
from sklearn.model_selection import train_test_split

raw_ascii = raw.assign(ascii=raw["name"].map(to_ascii), y=raw["language"].map({lang: k for k, lang in enumerate(LANGS)}))
r_tr, r_te = train_test_split(raw_ascii, test_size=0.17, stratify=raw_ascii["y"], random_state=0)
leak = make_pipeline(CountVectorizer(analyzer="char_wb", ngram_range=(1, 3)), LogisticRegression(max_iter=3000)).fit(r_tr["ascii"], r_tr["y"])
pred_leak = leak.predict(r_te["ascii"])
print(f"不去重、隨機切：accuracy {accuracy_score(r_te['y'], pred_leak):.4f}；test 裡有 {r_te['ascii'].isin(set(r_tr['ascii'])).mean():.1%} 的名字在 train 出現過")
per_lang_leak = pd.DataFrame({
    "leaky acc": pd.Series(pred_leak == r_te["y"].to_numpy(), index=r_te["language"]).groupby(level=0).mean(),
    "clean val acc": pd.Series(ngram_models[(1, 3)].predict(df_va["ascii"]) == y_va, index=df_va["language"]).groupby(level=0).mean(),
}).round(3)
display(per_lang_leak.sort_values("leaky acc", ascending=False).T)

In [ ]:
# TODO 2（問答）Baseline
#
# (1) 讀 val_table：
#     (a) Dummy 的 accuracy、macro-F1 各是多少？為什麼 macro-F1 這麼低？
#     (b) 1-gram（只有字母的次數）就能到多少？加上 2、3-gram（局部順序）之後，accuracy、macro-F1 各進步多少？
# 答：
#
# (2) 不去重的隨機切分：accuracy 高了多少？哪個語言被高估最多？為什麼？
#     原始教學的成績能直接拿來跟本章比嗎？
# 答：
#
# (3) 先寫下預測（第 4、6 節驗證，寫錯沒關係）：
#     手寫的 RNN（一個方向、H = 128）在 validation 上的 accuracy、macro-F1 會比 1～3-gram + LogReg 高還是低？
#     理由：RNN 能看「整個順序」，n-gram 只看 3 個字母的窗；但 RNN 要從 1 萬多個名字裡自己學會這些特徵。
# 答：

---
## 3. 手寫模型（NumPy）

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | `Embedding`：字母編號 → 向量（查表），反向傳播用 `np.add.at` | 3 |
| 3B | `RNN` 的 forward：隱藏狀態的遞迴、padding 與 mask | 4 |
| 3C | BPTT：時間上的反向傳播；梯度裁剪 | 5（推導）、6、7 |
| 3D | 組成 `CharRNNNumpy`、梯度檢查、訓練 | 8 |
| 3E | 實驗：梯度的消失與爆炸、讀的方向、逐字母的預測 | 9 |

### 從第 1 課搬過來的層（已完成）

`Linear`、`ReLU`、`Tanh`、`SoftmaxCrossEntropy`、初始化 `init_std`、梯度檢查 `numerical_grad`／`rel_error`／`check_layer` 都是第 1 課寫好的版本，原封不動（跟第 5 課 LeNet 用的同一份）。

In [ ]:
from typing import Literal, Self
from numpy.typing import ArrayLike
from scipy.special import logsumexp

InitName = Literal["he", "xavier"]


def init_std(init: InitName | float, fan_in: int, fan_out: int) -> float:
    """權重初始化的標準差。He：sqrt(2 / fan_in)（給 ReLU）；Xavier：sqrt(2 / (fan_in + fan_out))（給 tanh）。"""
    if isinstance(init, (int, float)):
        return float(init)
    if init == "he":
        return np.sqrt(2 / fan_in)
    if init == "xavier":
        return np.sqrt(2 / (fan_in + fan_out))
    raise ValueError(init)


class Layer:
    """所有層的共同介面。沒有參數的層，params／grads 是空 dict。"""

    def __init__(self) -> None:
        self.params: dict[str, np.ndarray] = {}
        self.grads: dict[str, np.ndarray] = {}

    def forward(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def backward(self, dout: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class Linear(Layer):
    """全連接層：out = X @ W + b。X (n, d_in)、W (d_in, d_out)、b (d_out,)。"""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        init: InitName | float = "he",
        rng: np.random.Generator | None = None,
        dtype: type = np.float64,
    ) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, init_std(init, d_in, d_out), size=(d_in, d_out))
        self.params = {"W": W.astype(dtype), "b": np.zeros(d_out, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return X @ self.params["W"] + self.params["b"]

    def backward(self, dout: np.ndarray) -> np.ndarray:
        self.grads["W"] = self.X.T @ dout
        self.grads["b"] = dout.sum(axis=0)
        return dout @ self.params["W"].T


class ReLU(Layer):
    """out = max(0, X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * self.mask


class Tanh(Layer):
    """out = tanh(X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.out = np.tanh(X)
        return self.out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * (1 - self.out**2)


class SoftmaxCrossEntropy:
    """softmax + 平均交叉熵。forward(Z, y) 回傳純量 loss；backward() 回傳 ∂L/∂Z = (P − Y)/n。"""

    def forward(self, Z: np.ndarray, y: np.ndarray) -> float:
        log_P = Z - logsumexp(Z, axis=1, keepdims=True)
        self.P = np.exp(log_P)
        self.y = y
        return float(-log_P[np.arange(len(y)), y].mean())

    def backward(self) -> np.ndarray:
        n = len(self.y)
        dZ = self.P.copy()
        dZ[np.arange(n), self.y] -= 1
        return dZ / n


def numerical_grad(f, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """中央差分：回傳與 x 同 shape 的 ∂f/∂x。會就地修改 x 再改回來。"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps
        f_plus = f()
        x[i] = old - eps
        f_minus = f()
        x[i] = old
        grad[i] = (f_plus - f_minus) / (2 * eps)
    return grad


def rel_error(a: np.ndarray, b: np.ndarray) -> float:
    """相對誤差 ‖a − b‖ / ‖a + b‖（兩者都是 0 時回傳 0）。"""
    denom = np.linalg.norm(a + b)
    return 0.0 if denom == 0 else float(np.linalg.norm(a - b) / denom)


def check_layer(layer: Layer, X: np.ndarray, seed: int = 0) -> dict[str, float]:
    """單一層的梯度檢查（float64）。f(X) = Σ forward(X) ⊙ G，G 是隨機的上游梯度。

    回傳 {"dX": 相對誤差, "dW": ..., "db": ...}（沒有參數的層只有 dX）。
    """
    G = np.random.default_rng(seed).normal(size=layer.forward(X).shape)
    f = lambda: float(np.sum(layer.forward(X) * G))
    layer.forward(X)
    dX = layer.backward(G)
    errs = {"dX": rel_error(dX, numerical_grad(f, X))}
    for k in layer.params:
        errs["d" + k] = rel_error(layer.grads[k], numerical_grad(f, layer.params[k]))
    return errs

### 3A. `Embedding`：字母 → 向量

第 1 課的 MLP 輸入是像素值，本身就是數字。字母不是數字：`a` = 1、`b` = 2 這種編號**沒有大小意義**（`b` 不是「兩倍的 `a`」）。

- **one-hot**：字母 $c$ → 長度 $V$ 的向量，第 $c$ 個位置是 1、其他是 0。再乘一個 $(V, E)$ 的矩陣 = **取出第 $c$ 列**（前置複習 7）。
- **Embedding 層**：直接存這個 $(V, E)$ 的矩陣 $W_{\text{emb}}$，forward 就是查表 `W[X]`。它是「輸入是 one-hot 的 `Linear`（沒有偏差）」，只是省掉乘一堆 0。

**反向傳播**：`Linear` 的 $\partial L/\partial W = X^\top \delta$；$X$ 是 one-hot 時，$X^\top \delta$ 的第 $c$ 列 = **所有「輸入是字母 $c$」的位置的 $\delta$ 加總**。同一個字母在一個 batch 裡出現很多次（`a` 幾乎每個名字都有）→ 必須**累加**。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 字母編號（0 = padding） | $(n, T)$ 整數 | `self.X` |
| $W_{\text{emb}}$ | 每個字母一列 | $(V, E)$ | `params["W"]` |
| out | $W_{\text{emb}}[X]$ | $(n, T, E)$ | forward 的回傳 |
| dout | $\partial L/\partial\,\text{out}$ | $(n, T, E)$ | `dout` |

**下標公式**：
- TODO 3：$\dfrac{\partial L}{\partial W_{c,e}} = \sum_{i, t:\ X_{it} = c}\text{dout}_{i,t,e}$ —— 加總所有「字母是 $c$」的位置 → `np.add.at(dW, self.X, dout)`（前置複習 8）。padding（$c = 0$）那一列的梯度設成 0，讓它永遠是零向量。

#### 最小例子：TODO 3（Embedding 的反向傳播）

$V = 4$、$E = 2$；一個名字 $X = [[1, 2, 1]]$（字母 1 出現兩次）；上游梯度 dout 是 $[[10, 10], [20, 20], [30, 30]]$。

1. **正確答案**：迴圈。字母 1 在位置 0、2 → $dW_1 = (10 + 30, 10 + 30) = (40, 40)$；字母 2 在位置 1 → $dW_2 = (20, 20)$；字母 0、3 沒出現 → 0。
2. **積木**：`dW[X] += dout` —— 看它錯在哪裡。
3. **留給你**：改成 `np.add.at`。

In [ ]:
# 最小例子：TODO 3（Embedding 的反向傳播）
X_mini = np.array([[1, 2, 1]])
dout_mini = np.array([[[10.0, 10.0], [20.0, 20.0], [30.0, 30.0]]])

# ---- 1. 正確答案：迴圈 ----
dW_loop = np.zeros((4, 2))
for i in range(X_mini.shape[0]):
    for t in range(X_mini.shape[1]):
        dW_loop[X_mini[i, t]] += dout_mini[i, t]
print("迴圈：\n", dW_loop)

# ---- 2. 積木：花式索引的 += ----
dW_wrong = np.zeros((4, 2))
dW_wrong[X_mini] += dout_mini
print("dW[X] += dout：\n", dW_wrong, "← 字母 1 只加到一次（後面的覆蓋前面的）")

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（第 1 列應為 [40 40]）：
# emb_mini = Embedding(4, 2); emb_mini.forward(X_mini); emb_mini.backward(dout_mini); print(emb_mini.grads["W"])

In [ ]:
class Embedding(Layer):
    """查表：out = W[X]。X (n, T) 是整數（0 = padding，那一列固定是 0）、W (V, E)、out (n, T, E)。"""

    def __init__(self, V: int, E: int, std: float = 0.1, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, std, size=(V, E))
        W[0] = 0.0
        self.params = {"W": W}
        self.grads = {"W": np.zeros_like(W)}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return self.params["W"][X]

    def backward(self, dout: np.ndarray) -> None:
        """回傳 None：輸入是整數編號，沒有「對輸入的梯度」。"""
        dW = np.zeros_like(self.params["W"])
        # TODO 3（程式）：把 dout 依 self.X 的編號累加到 dW（一行，見下標公式）
        raise NotImplementedError("TODO 3：Embedding 的反向傳播")
        dW[0] = 0.0  # padding 的向量不更新
        self.grads["W"] = dW
        return None

### 3B. `RNN` 的 forward

$$h_t = \tanh\big(x_t W_x + h_{t-1} W_h + b\big), \qquad h_0 = 0$$

- **同一組** $W_x, W_h, b$ 用在每一個時間步（參數共用）；
- $h_t$ 是「讀完前 $t$ 個字母」的摘要：$h_{t-1} W_h$ 把過去帶進來，$x_t W_x$ 加入新的字母；
- 分類只用**最後一個**隱藏狀態：讀完整個名字之後的 $h_{L}$ → `Linear` → softmax。

**padding 與 mask**：一個 batch 的名字長度不同，短的後面補 0（編號 0 的字母向量是零向量，但 $h_{t-1}W_h + b$ 不是 0，照算的話 $h$ 會繼續變）。所以在 padding 的位置**保持舊的 $h$**（前置複習 9）：
$$h_t = m_t \odot \tilde h_t + (1 - m_t) \odot h_{t-1}, \qquad \tilde h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$$
這樣讀到最後一格 $T$ 時，每個名字的 $h_T$ 剛好就是它自己的 $h_{L_i}$。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 字母向量（Embedding 的輸出） | $(n, T, E)$ | `X` |
| mask | 1 = 真的字母、0 = padding | $(n, T)$ | `mask` |
| $W_x$、$W_h$、$b$ | 參數 | $(E, H)$、$(H, H)$、$(H,)$ | `params["Wx"]`、`params["Wh"]`、`params["b"]` |
| $h_t$ | 第 $t$ 步之後的隱藏狀態（$t = 0$ 是初始值 0） | $(n, H)$ | `Hs[:, t]` |
| Hs | 全部的隱藏狀態 $h_0, \dots, h_T$（反向傳播要用） | $(n, T + 1, H)$ | `self.Hs` |

注意下標：迴圈變數 `t` 從 0 到 $T - 1$ 是「第 `t` 個字母」，讀完它之後的隱藏狀態存在 `Hs[:, t + 1]`。

**下標公式**：
- TODO 4：$\tilde h = \tanh(X_{:, t} W_x + H_{:, t} W_h + b)$ → `np.tanh(X[:, t] @ Wx + Hs[:, t] @ Wh + b)`；`Hs[:, t + 1] = m * h_new + (1 - m) * Hs[:, t]`，`m = mask[:, t, None]`。

#### 最小例子：TODO 4（$E = H = 1$ 的 RNN）

$W_x = 0.5$、$W_h = 1$、$b = 0$；兩個「名字」（每個字母向量只有 1 個數字）：$x^{(0)} = (1, 2)$、$x^{(1)} = (3, \text{pad})$。

1. **正確答案**（迴圈、純量）：名字 0：$h_1 = \tanh(0.5) = 0.4621$、$h_2 = \tanh(0.5 \cdot 2 + 0.4621) = \tanh(1.4621) = 0.8981$。名字 1：$h_1 = \tanh(1.5) = 0.9051$，第 2 步是 padding → $h_2 = h_1 = 0.9051$。
2. **積木**：一個時間步的矩陣版、mask 的廣播。
3. **留給你**：寫進 `RNN.forward` 的迴圈。

In [ ]:
# 最小例子：TODO 4（E = H = 1 的 RNN）
Xs_mini = np.array([[[1.0], [2.0]], [[3.0], [0.0]]])  # (n=2, T=2, E=1)
mask_mini = np.array([[1.0, 1.0], [1.0, 0.0]])
Wx_mini, Wh_mini, b_mini = np.array([[0.5]]), np.array([[1.0]]), np.array([0.0])

# ---- 1. 正確答案：迴圈、純量 ----
for i in range(2):
    h, hs = 0.0, []
    for t in range(2):
        h_new = np.tanh(0.5 * Xs_mini[i, t, 0] + 1.0 * h)
        h = h_new if mask_mini[i, t] else h
        hs.append(round(h, 4))
    print(f"名字 {i}：h_1, h_2 =", hs)

# ---- 2. 積木：第 1 步（t = 0）的矩陣版 ----
H0 = np.zeros((2, 1))
h_new0 = np.tanh(Xs_mini[:, 0] @ Wx_mini + H0 @ Wh_mini + b_mini)
m0 = mask_mini[:, 0, None]
print("\nX[:, 0] shape", Xs_mini[:, 0].shape, "；h_new shape", h_new0.shape, "；m shape", m0.shape, "\nh_1 =", (m0 * h_new0 + (1 - m0) * H0).ravel().round(4))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 [[0. 0.4621 0.8981] [0. 0.9051 0.9051]]）：
# rnn_mini = RNN(1, 1); rnn_mini.params.update(Wx=Wx_mini, Wh=Wh_mini, b=b_mini); rnn_mini.forward(Xs_mini, mask_mini); print(rnn_mini.Hs[:, :, 0].round(4))

### 3C. BPTT：時間上的反向傳播

把 RNN 沿著時間「攤開」（unroll），它就是一個 $T$ 層的網路，**每一層共用同一組參數**：

```
x_1        x_2        x_3              (每一步都用同一組 Wx、Wh、b)
 │          │          │
 ▼          ▼          ▼
h_0 ──▶ [ tanh ] ──▶ [ tanh ] ──▶ [ tanh ] ──▶ h_3 ──▶ Linear ──▶ softmax ──▶ L
           h_1        h_2
```

反向傳播從最後一步往回走。定義 $\delta_t = \partial L / \partial h_t$（$(n, H)$）。從輸出層拿到 $\delta_T$ 之後，每往回一步：

| 量 | 公式 | 說明 |
| --- | --- | --- |
| $a_t$ | $x_t W_x + h_{t-1}W_h + b$ | tanh 之前的值（forward 沒存，但 $\tanh'(a_t) = 1 - h_t^2$ 只需要 $h_t$） |
| $\partial L/\partial a_t$ | $\delta_t \odot m_t \odot (1 - h_t^2)$ | 前置複習 2；padding 的位置 $m_t = 0$ → 沒有梯度 |
| $\partial L/\partial W_x$ | $\mathrel{+}= x_t^\top\, \partial L/\partial a_t$ | 跟第 1 課 `Linear` 一樣，但**每一步都加上去**（參數共用 = 梯度加總，跟 LeNet 的濾波器一樣） |
| $\partial L/\partial W_h$ | $\mathrel{+}= h_{t-1}^\top\, \partial L/\partial a_t$ | 同上 |
| $\partial L/\partial b$ | $\mathrel{+}= \sum_i (\partial L/\partial a_t)_i$ | 同上 |
| $\partial L/\partial x_t$ | $(\partial L/\partial a_t)\, W_x^\top$ | 傳給 Embedding |
| $\delta_{t-1}$ | $(\partial L/\partial a_t)\, W_h^\top + (1 - m_t)\odot\delta_t$ | 兩條路：透過 $a_t$（真的字母）、或 padding 時 $h_t = h_{t-1}$ 直接傳回去 |

#### 符號表（3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\delta_t$ | $\partial L/\partial h_t$（上游梯度，一路往回傳） | $(n, H)$ | `dh` |
| $\partial L/\partial a_t$ | tanh 之前的梯度 | $(n, H)$ | `da` |
| $h_t$、$h_{t-1}$ | forward 存下的 cache | $(n, H)$ | `Hs[:, t + 1]`、`Hs[:, t]` |
| `grads[...]` | 參數的梯度（**結果**，一路加總） | 同參數 | `self.grads["Wx"]` 等 |
| dX | 傳給 Embedding 的梯度 | $(n, T, E)$ | `dX` |

In [ ]:
# TODO 5（問答）推導 BPTT（先在紙上算，再寫 TODO 6）
#
# 設定：一個名字、兩個字母（T = 2）、E = H = 1（全部是純量），沒有 padding。
#     a_1 = x_1 Wx + h_0 Wh + b，h_1 = tanh(a_1)；a_2 = x_2 Wx + h_1 Wh + b，h_2 = tanh(a_2)；h_0 = 0。
#     假設 loss 就是 L = h_2（所以 δ_2 = ∂L/∂h_2 = 1）。
#
# (1) ∂L/∂a_2 = ?（用 h_2 表示）
# 答：
#
# (2) Wh 出現在 a_1 和 a_2 兩個地方嗎？（看 a_1 的式子：h_0 = 0）
#     ∂L/∂Wh 有哪幾條路徑？每條路徑的貢獻是什麼？（前置複習 1：每條路徑加總）
# 答：
#
# (3) δ_1 = ∂L/∂h_1 = ?；∂L/∂a_1 = ?
# 答：
#
# (4) ∂L/∂Wx = x_2 · (∂L/∂a_2) + x_1 · (∂L/∂a_1)。代入 x_1 = 1、x_2 = 2、Wx = 0.5、Wh = 1、b = 0（同 TODO 4 的名字 0），
#     算出 ∂L/∂Wx、∂L/∂Wh、∂L/∂b 的數值。（下一格的最小例子會用數值微分驗證）
# 答：
#
# (5) 把 (3) 推廣到 T 步：δ_1 會是 δ_T 乘上幾個「(1 − h_t²) · Wh」？如果每個都是 0.5，T = 20 時 δ_1 大約是 δ_T 的幾倍？如果每個都是 1.5 呢？（前置複習 3）
# 答：

#### 最小例子：TODO 6（BPTT）

同 TODO 5：名字 0（$x = (1, 2)$）、$W_x = 0.5$、$W_h = 1$、$b = 0$、loss $= h_2$。

1. **正確答案**：數值微分（中央差分），每個參數各算一次。應為 $\partial L/\partial W_x \approx 0.5391$、$\partial L/\partial W_h \approx 0.0894$、$\partial L/\partial b \approx 0.3456$。
2. **積木**：只做**最後一步**（$t = 1$，第 2 個字母）：$\partial L/\partial a_2 = 1 - h_2^2$、它對 $W_x$ 的貢獻 $x_2 \cdot \partial L/\partial a_2$。
3. **留給你**：把每一步寫進迴圈、往回傳 $\delta$（TODO 6），對照 1。

In [ ]:
# 最小例子：TODO 6（BPTT）
x_seq = np.array([1.0, 2.0])


def loss_scalar(Wx: float, Wh: float, b: float) -> float:
    h = 0.0
    for x in x_seq:
        h = np.tanh(x * Wx + h * Wh + b)
    return h


# ---- 1. 正確答案：數值微分 ----
eps = 1e-6
num = {
    "Wx": (loss_scalar(0.5 + eps, 1.0, 0.0) - loss_scalar(0.5 - eps, 1.0, 0.0)) / (2 * eps),
    "Wh": (loss_scalar(0.5, 1.0 + eps, 0.0) - loss_scalar(0.5, 1.0 - eps, 0.0)) / (2 * eps),
    "b": (loss_scalar(0.5, 1.0, 0.0 + eps) - loss_scalar(0.5, 1.0, 0.0 - eps)) / (2 * eps),
}
print("數值微分：", {k: round(v, 4) for k, v in num.items()})

# ---- 2. 積木：只做最後一步 ----
h1, h2 = np.tanh(0.5), np.tanh(0.5 * 2 + np.tanh(0.5))
da2 = 1.0 * (1 - h2**2)
print(f"h_1 = {h1:.4f}、h_2 = {h2:.4f}；∂L/∂a_2 = {da2:.4f}；最後一步對 ∂L/∂Wx 的貢獻 x_2·∂L/∂a_2 = {2 * da2:.4f}（還差第 1 步的貢獻）")

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應跟上面的數值微分一樣）：
# r_ = RNN(1, 1); r_.params.update(Wx=np.array([[0.5]]), Wh=np.array([[1.0]]), b=np.array([0.0])); r_.forward(np.array([[[1.0], [2.0]]]), np.ones((1, 2))); r_.backward(np.ones((1, 1))); print({k: round(float(v.ravel()[0]), 4) for k, v in r_.grads.items()})

In [ ]:
class RNN(Layer):
    """單層 tanh RNN（batch_first）。forward(X, mask) 回傳最後的隱藏狀態 (n, H)。

    forward 存下的 cache：self.X (n, T, E)、self.mask (n, T)、self.Hs (n, T + 1, H)。
    backward 另外存 self.dh_norms (n, T)：‖∂L/∂h‖（讀完第 t 個字母後的隱藏狀態），實驗用。
    """

    def __init__(self, E: int, H: int, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, H)),
            "b": np.zeros(H),
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        Wx, Wh, b = self.params["Wx"], self.params["Wh"], self.params["b"]
        Hs = np.zeros((n, T + 1, Wh.shape[0]))
        for t in range(T):
            # TODO 4（程式）：算 h_new，再用 mask 決定 Hs[:, t + 1]（見下標公式）
            raise NotImplementedError("TODO 4：RNN forward")
        self.X, self.mask, self.Hs = X, mask, Hs
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，並把參數的梯度存進 self.grads。"""
        X, mask, Hs = self.X, self.mask, self.Hs
        n, T, _ = X.shape
        Wx, Wh = self.params["Wx"], self.params["Wh"]
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        dX = np.zeros_like(X)
        self.dh_norms = np.zeros((n, T))
        dh = dh_last
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t] = np.linalg.norm(dh, axis=1)
            # TODO 6（程式）：BPTT 的一步（見 3C 的表格，由上往下照抄成程式）
            #   - m = mask[:, t, None]；h = Hs[:, t + 1]；h_prev = Hs[:, t]
            #   - da = dh * m * (1 − h²)
            #   - grads["Wx"] += X[:, t].T @ da；grads["Wh"] += h_prev.T @ da；grads["b"] += da.sum(axis=0)
            #   - dX[:, t] = da @ Wx.T
            #   - dh = da @ Wh.T + dh * (1 − m)      ← 常見錯誤：忘了 padding 那條路，或用 h 而不是 h_prev 算 grads["Wh"]
            raise NotImplementedError("TODO 6：BPTT")
        return dX

#### 梯度裁剪（gradient clipping）

TODO 5 (5)：梯度可能隨 $T$ 指數爆炸。一個爆炸的 batch 就能把參數推到很遠的地方（loss 變成 nan）。**裁剪**：把所有參數的梯度看成一條長向量，全域範數（前置複習 5）超過門檻 $c$ 就等比例縮小：
$$g \leftarrow g \cdot \min\Big(1, \frac{c}{\lVert g\rVert}\Big)$$
方向不變、只限制長度。（PyTorch：`torch.nn.utils.clip_grad_norm_`。）

**最小例子**：兩個參數的梯度 $(3, 4)$ 和 $(0,)$ → 全域範數 5；門檻 1 → 縮成 $(0.6, 0.8)$；門檻 10 → 不變。

In [ ]:
def clip_grad_norm(grads: list[np.ndarray], max_norm: float) -> float:
    """就地把梯度縮放到全域範數 ≤ max_norm。回傳縮放前的全域範數。"""
    # TODO 7（程式）：
    #   - total = sqrt(所有 grads 的平方和加總)
    #   - scale = min(1, max_norm / total)（total = 0 時 scale = 1）
    #   - 每個 g 就地乘上 scale（g *= scale；不要寫 g = g * scale —— 那只是改了迴圈變數，list 裡的陣列沒變）
    raise NotImplementedError("TODO 7：梯度裁剪")
    return total


# 寫完 TODO 7 之後，取消下面的註解來對照（應為 5.0 [0.6 0.8]）：
# g_mini = [np.array([3.0, 4.0]), np.array([0.0])]; print(clip_grad_norm(g_mini, 1.0), g_mini[0])

### 3D. 組成 `CharRNNNumpy`、梯度檢查、訓練

**程式結構圖**（誰身上有什麼）：

```
CharRNNNumpy
├── emb  : Embedding   params {"W": (V, E)}          cache: X (n, T)
├── rnn  : RNN          params {"Wx", "Wh", "b"}      cache: X (n, T, E)、mask、Hs (n, T+1, H)
├── fc   : Linear       params {"W": (H, K), "b"}     cache: X (n, H)  ← 這裡的 X 是 h_T
├── loss : SoftmaxCrossEntropy                        cache: P、y
└── layers = [emb, rnn, fc]   ← 走過所有參數時用（_param_list）
```

- `forward(Xi, mask)`：`emb.forward(Xi)` → `rnn.forward(E, mask)` → `fc.forward(h_T)` → logits。
- `loss_and_backward(Xi, mask, y)`：forward → `loss.forward` → `loss.backward()` → `fc.backward` → `rnn.backward` → `emb.backward`。
- `fit`：mini-batch SGD + 梯度裁剪（TODO 7）。batch 依長度排序後切（減少 padding），每個 epoch 打亂 batch 的順序；每個 batch 只取到它最長的名字（`[:, :L_max]`）。每個 epoch 量 validation，最後換回 validation loss 最低的那個 epoch 的權重。

**資料的編碼**：`encode(names)` → `Xi` $(n, T_{\max})$ 整數、`mask` $(n, T_{\max})$。

In [ ]:
def encode(names: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """名字的清單 → (字母編號 (n, T_max)，mask (n, T_max) 0/1)。T_max = 最長的名字長度。"""
    T = max(map(len, names))
    Xi = np.zeros((len(names), T), dtype=np.int64)
    mask = np.zeros((len(names), T))
    for i, nm in enumerate(names):
        Xi[i, : len(nm)] = [CHAR_TO_ID[c] for c in nm]
        mask[i, : len(nm)] = 1.0
    return Xi, mask


Xi_tr, M_tr = encode(df_tr["ascii"].tolist())
Xi_va, M_va = encode(df_va["ascii"].tolist())
Xi_te, M_te = encode(df_te["ascii"].tolist())
print("Xi_tr", Xi_tr.shape, "；padding 佔", f"{1 - M_tr.mean():.1%}", "（不排序、整個訓練集一起補到最長）")


class CharRNNNumpy:
    """Embedding → RNN → Linear → softmax。屬性名稱仿 sklearn：history_、best_epoch_。"""

    def __init__(self, V: int, K: int, E: int = 32, H: int = 128, lr: float = 0.1, batch_size: int = 64,
                 n_epochs: int = 30, clip: float | None = 5.0, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.emb = Embedding(V, E, rng=rng)
        self.rnn = RNN(E, H, rng=rng)
        self.fc = Linear(H, K, init="xavier", rng=rng)
        self.loss = SoftmaxCrossEntropy()
        self.layers = [self.emb, self.rnn, self.fc]
        self.lr, self.batch_size, self.n_epochs, self.clip, self.seed = lr, batch_size, n_epochs, clip, seed

    def _param_list(self) -> list[tuple[dict, dict, str]]:
        return [(layer.params, layer.grads, k) for layer in self.layers for k in layer.params]

    def forward(self, Xi: np.ndarray, mask: np.ndarray) -> np.ndarray:
        """logits (n, K)。"""
        return self.fc.forward(self.rnn.forward(self.emb.forward(Xi), mask))

    def loss_and_backward(self, Xi: np.ndarray, mask: np.ndarray, y: np.ndarray) -> float:
        loss = self.loss.forward(self.forward(Xi, mask), y)
        self.emb.backward(self.rnn.backward(self.fc.backward(self.loss.backward())))
        return loss

    def predict_proba(self, Xi: np.ndarray, mask: np.ndarray, batch_size: int = 2048) -> np.ndarray:
        out = []
        for s in range(0, len(Xi), batch_size):
            Z = self.forward(Xi[s : s + batch_size], mask[s : s + batch_size])
            Z = Z - Z.max(axis=1, keepdims=True)
            P = np.exp(Z)
            out.append(P / P.sum(axis=1, keepdims=True))
        return np.vstack(out)

    def fit(self, Xi: np.ndarray, mask: np.ndarray, y: np.ndarray, Xi_val: np.ndarray | None = None,
            mask_val: np.ndarray | None = None, y_val: np.ndarray | None = None, verbose: bool = True) -> "CharRNNNumpy":
        rng = np.random.default_rng(self.seed + 1)
        order = np.argsort(mask.sum(axis=1), kind="stable")
        batches = [order[s : s + self.batch_size] for s in range(0, len(order), self.batch_size)]
        self.history_ = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "grad_norm": [], "seconds": []}
        best_loss, best_state = np.inf, None
        for epoch in range(self.n_epochs):
            t0, total, norms = time.perf_counter(), 0.0, []
            for b in rng.permutation(len(batches)):
                idx = batches[b]
                L = int(mask[idx].sum(axis=1).max())
                total += self.loss_and_backward(Xi[idx, :L], mask[idx, :L], y[idx]) * len(idx)
                grads = [g[k] for _, g, k in self._param_list()]
                norms.append(clip_grad_norm(grads, self.clip) if self.clip else float(np.sqrt(sum((g**2).sum() for g in grads))))
                for p, g, k in self._param_list():
                    p[k] -= self.lr * g[k]
            self.history_["train_loss"].append(total / len(y))
            self.history_["grad_norm"].append(float(np.median(norms)))
            msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
            if not np.isfinite(total):
                if verbose:
                    print(msg + "  → 發散，停止訓練")
                break
            if Xi_val is not None:
                s = val_scores(self.predict_proba(Xi_val, mask_val), y_val)
                self.history_["val_loss"].append(s["log_loss"])
                self.history_["val_acc"].append(s["accuracy"])
                self.history_["val_f1"].append(s["macro_f1"])
                msg += f"  val_loss {s['log_loss']:.4f}  val_acc {s['accuracy']:.4f}  val_f1 {s['macro_f1']:.4f}"
                if s["log_loss"] < best_loss:
                    best_loss, self.best_epoch_ = s["log_loss"], epoch + 1
                    best_state = [(p, k, p[k].copy()) for p, _, k in self._param_list()]
            self.history_["seconds"].append(time.perf_counter() - t0)
            if verbose:
                print(msg + f"  ({self.history_['seconds'][-1]:.1f}s)")
        if best_state is not None:
            for p, k, v in best_state:
                p[k] = v
        return self

In [ ]:
# 整個網路的梯度檢查：小網路（V = 6、E = 3、H = 4、K = 3），3 個長度不同的名字（含 padding）
gc = CharRNNNumpy(V=6, K=3, E=3, H=4, seed=1)
Xi_gc = np.array([[1, 2, 3, 4], [5, 1, 0, 0], [2, 2, 2, 0]])
M_gc = (Xi_gc > 0).astype(float)
y_gc = np.array([0, 2, 1])
gc.loss_and_backward(Xi_gc, M_gc, y_gc)
f_gc = lambda: gc.loss.forward(gc.forward(Xi_gc, M_gc), y_gc)
for layer, name in [(gc.emb, "emb"), (gc.rnn, "rnn"), (gc.fc, "fc")]:
    for k in layer.params:
        analytic = layer.grads[k].copy()
        numeric = numerical_grad(f_gc, layer.params[k])
        if name == "emb":
            numeric[0] = 0.0  # padding 那一列：我們刻意不更新
        print(f"✅ {name}.{k:2s} 相對誤差 {rel_error(analytic, numeric):.1e}（應 < 1e-6）")

# padding 不影響結果：同一個名字，單獨算 vs 放在補過 padding 的 batch 裡
z_alone = gc.forward(Xi_gc[1:2, :2], M_gc[1:2, :2])
z_batch = gc.forward(Xi_gc, M_gc)[1:2]
print("✅ padding 不改變結果？", np.allclose(z_alone, z_batch))

#### 第一個 RNN：$E = 32$、$H = 128$、SGD lr = 0.1、batch 64、梯度裁剪 5、30 epochs

（這台電腦一個 epoch 約 1 秒。）

In [ ]:
rnn_np = CharRNNNumpy(V, K, E=32, H=128, lr=0.1, n_epochs=30, clip=5.0, seed=0)
rnn_np.fit(Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va)
val_table.loc["RNN (scratch NumPy, SGD)"] = val_scores(rnn_np.predict_proba(Xi_va, M_va))
print("best epoch：", rnn_np.best_epoch_)
display(val_table)

fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
ep = np.arange(1, len(rnn_np.history_["train_loss"]) + 1)
axes[0].plot(ep, rnn_np.history_["train_loss"], label="train")
axes[0].plot(ep, rnn_np.history_["val_loss"], label="val")
axes[0].set_title("cross-entropy")
axes[0].legend()
axes[1].plot(ep, rnn_np.history_["val_acc"], label="val accuracy")
axes[1].plot(ep, rnn_np.history_["val_f1"], label="val macro-F1")
axes[1].legend()
axes[2].plot(ep, rnn_np.history_["grad_norm"])
axes[2].axhline(5, color="gray", ls=":")
axes[2].set_title("median global grad norm (before clipping)")
for ax in axes:
    ax.set_xlabel("epoch")
plt.show()

In [ ]:
# TODO 8（問答）訓練結果
#
# (1) 梯度檢查：每個參數的相對誤差都 < 1e-6 嗎？emb 的第 0 列為什麼要特別處理？
# 答：
#
# (2) 讀 val_table 與學習曲線：
#     (a) 手寫 RNN 的 validation accuracy、macro-F1 是多少？跟 TODO 2 (3) 的預測一樣嗎？比 1～3-gram + LogReg 好還是差？
#     (b) train loss 和 val loss 的差距：有過擬合嗎？從第幾個 epoch 開始？
#     (c) 右圖：梯度的範數有多常超過裁剪門檻 5？如果把 clip 拿掉，你預期會發生什麼？（第 3E 節的實驗 2 會試）
# 答：

### 3E. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 梯度的消失與爆炸：$\lVert\partial L/\partial h_t\rVert$ 隨「離最後一個字母幾步」怎麼變？$W_h$ 的大小（譜半徑）有什麼影響？ |
| 2 | 拿掉梯度裁剪、用大的學習率會怎樣？ |
| 3 | 讀的方向：把名字**反過來**讀（`Ivanov` → `vonavI`），accuracy 會變好還是變差？ |
| 4 | 逐字母的預測：讀到第幾個字母時，模型就「確定」了？ |

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| 譜半徑 $\rho(W_h)$ | 前置複習 4。實驗 1 把 $W_h$ 換成「正交矩陣 × $\rho$」，其他參數不變 |
| 離最後幾步 | $L_i - 1 - t$：0 = 最後一個字母 |

In [ ]:
# 實驗 1：梯度的消失與爆炸（validation 裡長度 ≥ 12 的名字）
long_idx = np.flatnonzero(M_va.sum(axis=1) >= 12)[:300]
Xi_long, M_long = Xi_va[long_idx], M_va[long_idx]
L_long = M_long.sum(axis=1).astype(int)


def grad_by_distance(model: CharRNNNumpy) -> np.ndarray:
    """回傳 (12,)：離最後一個字母 0, 1, ..., 11 步的平均 ‖∂L/∂h‖（除以「離最後 0 步」的值）。"""
    model.loss_and_backward(Xi_long, M_long, y_va[long_idx])
    out = np.zeros(12)
    for d in range(12):
        out[d] = np.mean([model.rnn.dh_norms[i, L_long[i] - 1 - d] for i in range(len(L_long))])
    return out / out[0]


Q_orth, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(128, 128)))
curves = {"trained (rnn_np)": grad_by_distance(rnn_np)}
for rho in [0.5, 1.0, 1.5, 3.0]:
    m = CharRNNNumpy(V, K, E=32, H=128, seed=0)
    m.rnn.params["Wh"] = Q_orth * rho
    curves[f"untrained, Wh = {rho} × orthogonal"] = grad_by_distance(m)
fig, ax = plt.subplots(figsize=(8, 4))
for name, c in curves.items():
    ax.semilogy(np.arange(12), c, "o-", label=name)
ax.set_xlabel("steps before the last character")
ax.set_ylabel("‖∂L/∂h‖ (relative to last step)")
ax.legend(fontsize=8)
plt.show()
print({k: f"{v[-1]:.1e}" for k, v in curves.items()}, "← 離最後 11 步")
print("訓練後 Wh 的譜半徑：", round(float(np.abs(np.linalg.eigvals(rnn_np.rnn.params["Wh"])).max()), 3),
      "；h 飽和（|h| > 0.95）的比例：", round(float((np.abs(rnn_np.rnn.Hs[:, 1:]) > 0.95).mean()), 3))

In [ ]:
# 實驗 2：大的學習率，有／沒有梯度裁剪（各 8 個 epoch）
clip_rows = {}
for lr, clip in [(0.5, None), (0.5, 5.0), (1.0, None), (1.0, 5.0)]:
    m = CharRNNNumpy(V, K, lr=lr, clip=clip, n_epochs=8, seed=0).fit(Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va, verbose=False)
    h = m.history_
    clip_rows[f"lr={lr}, clip={clip}"] = {"max train loss": max(h["train_loss"]), "last train loss": h["train_loss"][-1],
                                          "max median grad norm": max(h["grad_norm"]), "best val acc": max(h["val_acc"]) if h["val_acc"] else np.nan}
display(pd.DataFrame(clip_rows).T.round(3))

In [ ]:
# 實驗 3：讀的方向（反過來讀：'Ivanov' → 'vonavI'），其他設定相同、各 20 epochs
def encode_reversed(names: list[str]) -> tuple[np.ndarray, np.ndarray]:
    return encode([nm[::-1] for nm in names])


Xi_tr_r, M_tr_r = encode_reversed(df_tr["ascii"].tolist())
Xi_va_r, M_va_r = encode_reversed(df_va["ascii"].tolist())
dir_rows = {}
for name, (a, b, c, d) in {"forward (Ivanov)": (Xi_tr, M_tr, Xi_va, M_va), "reversed (vonavI)": (Xi_tr_r, M_tr_r, Xi_va_r, M_va_r)}.items():
    m = CharRNNNumpy(V, K, n_epochs=20, seed=0).fit(a, b, y_tr, c, d, y_va, verbose=False)
    dir_rows[name] = val_scores(m.predict_proba(c, d))
    if name.startswith("reversed"):
        rnn_np_rev = m
display(pd.DataFrame(dir_rows).T)

In [ ]:
# 實驗 4：逐字母的預測 —— 每讀一個字母，就用當下的 h_t 預測一次
def prefix_probs(model: CharRNNNumpy, name: str) -> np.ndarray:
    """回傳 (L, K)：讀完第 1, 2, ..., L 個字母時的類別機率。"""
    Xi, M = encode([name])
    model.forward(Xi, M)
    Z = model.rnn.Hs[0, 1:] @ model.fc.params["W"] + model.fc.params["b"]
    Z = Z - Z.max(axis=1, keepdims=True)
    P = np.exp(Z)
    return P / P.sum(axis=1, keepdims=True)


demo_names = ["Nakamura", "Kowalski", "Gonzalez", "Ivanov", "Schneider", "OConnell"]
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, nm in zip(axes.ravel(), demo_names):
    P = prefix_probs(rnn_np, nm)
    top = np.argsort(P[-1])[::-1][:4]
    for k in top:
        ax.plot(np.arange(1, len(nm) + 1), P[:, k], "o-", label=LANGS[k])
    ax.set_xticks(np.arange(1, len(nm) + 1), list(nm))
    ax.set_ylim(0, 1)
    ax.set_title(nm)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
# TODO 9（問答）實驗 1–4
#
# (1) 實驗 1：
#     (a) 「Wh = 0.5 × 正交」的曲線：離最後 11 步時，梯度剩下最後一步的多少？「3.0 × 正交」呢？
#     (b) 「3.0 × 正交」理論上應該爆炸（3^11），為什麼實際上沒有那麼大？（提示：TODO 5 (5) 的每一步倍數是 (1 − h²)·Wh；ρ 很大時 h 會怎樣？看「飽和」）
#     (c) 訓練好的模型：梯度傳到 11 步之前剩多少？這代表「名字開頭的字母」對參數更新的影響大還是小？
# 答：
#
# (2) 實驗 2：
#     (a) 四種設定的 train loss 都比「隨機猜」（ln 18 ≈ 2.89）高嗎？跟 lr = 0.1 的第一個模型比呢？
#     (b) 在這裡，裁剪有讓結果變好嗎？為什麼 lr 太大時，裁剪也救不了？
#         （提示：裁剪只保證「每一步的位移 ≤ lr × 門檻」；lr = 1、門檻 5 時，一步最多可以走多遠？跟權重本身的大小比呢？）
#     (c) 結論：梯度裁剪是用來防止「偶爾一個爆炸的 batch」，還是用來讓「太大的學習率」變得可以用？
# 答：
#
# (3) 實驗 3：反過來讀，accuracy 變好還是變差？用 EDA 的字尾表和實驗 1 解釋：
#     正著讀時，字尾是「最後讀到的」→ 離輸出近；反著讀時，字尾是「最先讀到的」→ 要撐過整個名字的遞迴才到輸出。
#     如果兩種方向的結果差不多，那代表什麼？（提示：名字很短，平均 7 個字母）
# 答：
#
# (4) 實驗 4：
#     (a) 每個名字讀到第幾個字母時，正確的語言變成第一名？是開頭、中間還是結尾的字母讓機率跳最多？
#     (b) 「Kowalski」讀到「Kowal」時模型猜什麼？讀到最後的「-ski」之後呢？
# 答：

---
## 4. 套件實作：PyTorch `nn.RNN`

| PyTorch | 對應手寫的 | 注意 |
| --- | --- | --- |
| `nn.Embedding(V, E, padding_idx=0)` | `Embedding` | `padding_idx` 那一列不更新（我們的 `dW[0] = 0`） |
| `nn.RNN(E, H, batch_first=True, nonlinearity="tanh")` | `RNN` | 權重存成**轉置**：`weight_ih_l0` = $W_x^\top$ $(H, E)$、`weight_hh_l0` = $W_h^\top$；偏差有**兩個**（`bias_ih_l0` + `bias_hh_l0`，我們只有一個 $b$ = 兩者之和） |
| `pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)` | mask | 告訴 RNN 每個序列的真正長度；回傳的 `h_n` 就是每個序列**自己的**最後一步（不用 mask） |
| `nn.Linear(H, K)` | `Linear` | `weight` = $W^\top$ |
| `nn.utils.clip_grad_norm_(params, 5)` | `clip_grad_norm` | |
| `bidirectional=True` | — | 另一個 RNN 從右往左讀；`h_n` 有兩個方向，串起來 → $2H$ |

**本機用 CPU**：這個模型很小（約 2 萬個參數）、每個 batch 只有 64 個短序列，GPU（MPS／CUDA）啟動每個運算的固定開銷比計算本身還大，通常反而比 CPU 慢。想試 GPU 可以把下一格的 `device` 改成 `get_device()`。

In [ ]:
import torch
from torch import nn
from learning_map.utils import count_parameters, get_device, set_seed

device = torch.device("cpu")


class CharRNNTorch(nn.Module):
    """Embedding → nn.RNN（或 Bi-RNN）→ Linear。forward(X, lengths) → logits (n, K)。"""

    def __init__(self, V: int, K: int, E: int = 32, H: int = 128, bidirectional: bool = False) -> None:
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.rnn = nn.RNN(E, H, batch_first=True, nonlinearity="tanh", bidirectional=bidirectional)
        self.fc = nn.Linear(H * (2 if bidirectional else 1), K)

    def forward(self, X: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(X), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)  # h_n (方向數, n, H)：每個序列自己的最後一步
        h = torch.cat([h_n[0], h_n[1]], dim=1) if self.rnn.bidirectional else h_n[0]
        return self.fc(h)


def load_from_numpy(net: CharRNNTorch, model: CharRNNNumpy) -> CharRNNTorch:
    """把手寫模型的權重複製進 PyTorch 模型（注意轉置與兩個偏差）。"""
    with torch.no_grad():
        net.emb.weight.copy_(torch.as_tensor(model.emb.params["W"]))
        net.rnn.weight_ih_l0.copy_(torch.as_tensor(model.rnn.params["Wx"].T))
        net.rnn.weight_hh_l0.copy_(torch.as_tensor(model.rnn.params["Wh"].T))
        net.rnn.bias_ih_l0.copy_(torch.as_tensor(model.rnn.params["b"]))
        net.rnn.bias_hh_l0.zero_()
        net.fc.weight.copy_(torch.as_tensor(model.fc.params["W"].T))
        net.fc.bias.copy_(torch.as_tensor(model.fc.params["b"]))
    return net

#### 核對：手寫的 forward／BPTT = PyTorch autograd

同樣的權重（float64）、同一個 batch（含 padding），比 logits、loss 與每個參數的梯度。

In [ ]:
idx_chk = np.arange(64)
Xi_chk, M_chk = Xi_tr[idx_chk], M_tr[idx_chk]
L_chk = int(M_chk.sum(axis=1).max())
Xi_chk, M_chk = Xi_chk[:, :L_chk], M_chk[:, :L_chk]
loss_np = rnn_np.loss_and_backward(Xi_chk, M_chk, y_tr[idx_chk])

net_chk = load_from_numpy(CharRNNTorch(V, K).double(), rnn_np)
logits_t = net_chk(torch.as_tensor(Xi_chk), torch.as_tensor(M_chk.sum(axis=1).astype(np.int64)))
loss_t = nn.functional.cross_entropy(logits_t, torch.as_tensor(y_tr[idx_chk]))
loss_t.backward()
print(f"✅ loss：手寫 {loss_np:.10f}、PyTorch {loss_t.item():.10f}")
pairs = {
    "emb.W": (rnn_np.emb.grads["W"], net_chk.emb.weight.grad.numpy()),
    "rnn.Wx": (rnn_np.rnn.grads["Wx"], net_chk.rnn.weight_ih_l0.grad.numpy().T),
    "rnn.Wh": (rnn_np.rnn.grads["Wh"], net_chk.rnn.weight_hh_l0.grad.numpy().T),
    "rnn.b": (rnn_np.rnn.grads["b"], net_chk.rnn.bias_ih_l0.grad.numpy()),
    "fc.W": (rnn_np.fc.grads["W"], net_chk.fc.weight.grad.numpy().T),
}
for k, (a, b) in pairs.items():
    print(f"✅ {k:7s} 梯度最大差異 {np.abs(a - b).max():.1e}（應 < 1e-10）")
print("   PyTorch 的兩個偏差梯度一樣嗎？", torch.allclose(net_chk.rnn.bias_ih_l0.grad, net_chk.rnn.bias_hh_l0.grad))

#### 用 PyTorch 訓練

`train_seq` 跟第 1 課的 `train_torch` 做同樣的事（每個 epoch 打亂、量 validation、記 history、`restore_best` 換回 validation loss 最低的權重），差別只有：

- 輸入是 `(Xi, lengths)`，每個 batch 只取到它最長的名字；
- 最佳化器用 **Adam**（自動調整每個參數的步長；先當成「比 SGD 好調的 SGD」，細節在待補強），預設 lr = 3e-3；
- 每一步都做梯度裁剪（門檻 5）；
- 可以給 `class_weight`（每一類的 loss 權重，第 5 節用）。

In [ ]:
@torch.no_grad()
def predict_proba_seq(net: nn.Module, Xi: np.ndarray, mask: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    """(n, K) 的類別機率（float64 numpy）。"""
    net.eval()
    out = []
    for s in range(0, len(Xi), batch_size):
        L = mask[s : s + batch_size].sum(axis=1).astype(np.int64)
        X = torch.as_tensor(Xi[s : s + batch_size, : L.max()], device=device)
        out.append(torch.softmax(net(X, torch.as_tensor(L)).double(), dim=1).cpu().numpy())
    return np.vstack(out)


def train_seq(net: nn.Module, Xi: np.ndarray, mask: np.ndarray, y: np.ndarray, Xi_val: np.ndarray | None = None,
              mask_val: np.ndarray | None = None, y_val: np.ndarray | None = None, *, lr: float = 3e-3, batch_size: int = 64,
              n_epochs: int = 20, weight_decay: float = 0.0, class_weight: np.ndarray | None = None, clip: float = 5.0,
              seed: int = 0, restore_best: bool = True, verbose: bool = True) -> dict:
    """Adam + 梯度裁剪訓練 net（就地修改）。history 同第 1 課（另加 val_f1）。"""
    set_seed(seed)
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    w = None if class_weight is None else torch.as_tensor(class_weight, dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    lengths = mask.sum(axis=1).astype(np.int64)
    rng = np.random.default_rng(seed)
    hist: dict = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "seconds": []}
    best_loss, best_epoch, best_state = np.inf, 0, None
    for epoch in range(n_epochs):
        net.train()
        t0, total = time.perf_counter(), 0.0
        perm = rng.permutation(len(y))
        for s in range(0, len(y), batch_size):
            idx = perm[s : s + batch_size]
            L = lengths[idx]
            X = torch.as_tensor(Xi[idx, : L.max()], device=device)
            loss = loss_fn(net(X, torch.as_tensor(L)), torch.as_tensor(y[idx], device=device))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
            total += loss.item() * len(idx)
        hist["train_loss"].append(total / len(y))
        msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
        if Xi_val is not None:
            s_ = val_scores(predict_proba_seq(net, Xi_val, mask_val), y_val)
            hist["val_loss"].append(s_["log_loss"])
            hist["val_acc"].append(s_["accuracy"])
            hist["val_f1"].append(s_["macro_f1"])
            msg += f"  val_loss {s_['log_loss']:.4f}  val_acc {s_['accuracy']:.4f}  val_f1 {s_['macro_f1']:.4f}"
            if s_["log_loss"] < best_loss:
                best_loss, best_epoch = s_["log_loss"], epoch + 1
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        hist["seconds"].append(time.perf_counter() - t0)
        if verbose:
            print(msg + f"  ({hist['seconds'][-1]:.1f}s)")
    hist["best_epoch"], hist["best_val_loss"] = best_epoch, best_loss
    if restore_best and best_state is not None:
        net.load_state_dict(best_state)
    return hist


set_seed(0)
net_rnn = CharRNNTorch(V, K)
print("參數量：", count_parameters(net_rnn))
hist_rnn = train_seq(net_rnn, Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va)
val_table.loc["RNN (PyTorch, Adam)"] = val_scores(predict_proba_seq(net_rnn, Xi_va, M_va))

set_seed(0)
net_birnn = CharRNNTorch(V, K, bidirectional=True)
hist_birnn = train_seq(net_birnn, Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va, verbose=False)
val_table.loc["Bi-RNN (PyTorch, Adam)"] = val_scores(predict_proba_seq(net_birnn, Xi_va, M_va))
print("Bi-RNN best epoch：", hist_birnn["best_epoch"], "；每個 epoch", f"{np.mean(hist_birnn['seconds']):.1f}s", "（單向", f"{np.mean(hist_rnn['seconds']):.1f}s）")
display(val_table)

In [ ]:
# TODO 10（問答）PyTorch
#
# (1) 核對：loss 與每個梯度的差異是多少？PyTorch 的 bias_ih 和 bias_hh 的梯度為什麼一模一樣？（提示：兩者在公式裡的位置）
#     那為什麼 PyTorch 要放兩個偏差？（提示：cuDNN 的慣例；對模型的表達能力沒有差別）
# 答：
#
# (2) pack_padded_sequence 取代了我們的 mask。如果「不 pack、直接把補過 0 的 batch 丟進 nn.RNN」，h_n 會是什麼？（提示：最後一步是 padding）
# 答：
#
# (3) SGD（手寫）vs Adam（PyTorch）：同樣的模型，validation 成績差多少？達到最好的 epoch 各是第幾個？
# 答：
#
# (4) Bi-RNN：
#     (a) 參數量、每個 epoch 的時間大約變幾倍？成績進步多少？
#     (b) 用實驗 3（讀的方向）解釋：Bi-RNN 同時有「正著讀」和「反著讀」的 h，它的最後一個 h 同時「靠近」名字的哪兩端？
# 答：

---
## 5. 調參

依 README 的調參流程，**全程只用 train／validation**。這一章的超參數：

| 類型 | 超參數 | 範圍 |
| --- | --- | --- |
| 容量 | 隱藏層大小 $H$ | 32、64、128、256（log 尺度） |
| 容量 | 方向 | 單向、雙向 |
| 正則化／不平衡 | class weight | 無、「平衡」（第 $k$ 類的權重 $= n / (K \cdot n_k)$，小類別權重大） |
| 最佳化 | lr、epoch | Adam lr = 3e-3；epoch 用 early stopping（`restore_best`） |

**選模型的指標**：validation **macro-F1**（每一類同等重要，TODO 1 (1)），accuracy、log loss 一起看。

In [ ]:
# 5A. H 的 validation curve（單向、20 epochs、restore_best）
rows = []
for H_ in [32, 64, 128, 256]:
    set_seed(0)
    net = CharRNNTorch(V, K, H=H_)
    h = train_seq(net, Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va, verbose=False)
    s = val_scores(predict_proba_seq(net, Xi_va, M_va))
    tr_s = val_scores(predict_proba_seq(net, Xi_tr, M_tr), y_tr)
    rows.append({"H": H_, "params": count_parameters(net), "train acc": tr_s["accuracy"], "val acc": s["accuracy"], "val macro-F1": s["macro_f1"],
                 "val loss": s["log_loss"], "best epoch": h["best_epoch"], "sec/epoch": np.mean(h["seconds"])})
curve_H = pd.DataFrame(rows).set_index("H")
display(curve_H.round(4))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(curve_H.index, curve_H["train acc"], "o-", base=2, label="train acc")
ax.semilogx(curve_H.index, curve_H["val acc"], "o-", base=2, label="val acc")
ax.semilogx(curve_H.index, curve_H["val macro-F1"], "o-", base=2, label="val macro-F1")
ax.set_xlabel("H")
ax.legend()
plt.show()

In [ ]:
# 5B. 2 × 2：方向 × class weight（H = 128）
counts_tr = np.bincount(y_tr, minlength=K)
balanced_w = len(y_tr) / (K * counts_tr)
grid = []
for bi in [False, True]:
    for cw in [None, "balanced"]:
        set_seed(0)
        net = CharRNNTorch(V, K, bidirectional=bi)
        h = train_seq(net, Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va, class_weight=None if cw is None else balanced_w, verbose=False)
        grid.append({"bidirectional": bi, "class_weight": str(cw), **val_scores(predict_proba_seq(net, Xi_va, M_va)), "best_epoch": h["best_epoch"]})
grid = pd.DataFrame(grid)
display(grid)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
    sns.heatmap(grid.pivot(index="bidirectional", columns="class_weight", values=metric), annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title("val " + metric)
plt.show()

In [ ]:
# 5C. seed 造成的雜訊：5B 裡 macro-F1 最高的設定，換 3 個 seed 重跑
best_row = grid.loc[grid["macro_f1"].idxmax()]
best_cfg = {"bidirectional": bool(best_row["bidirectional"]), "class_weight": None if best_row["class_weight"] == "None" else "balanced"}
print("5B 最好的設定：", best_cfg)
seed_rows = []
for seed in [0, 1, 2]:
    set_seed(seed)
    net = CharRNNTorch(V, K, bidirectional=best_cfg["bidirectional"])
    h = train_seq(net, Xi_tr, M_tr, y_tr, Xi_va, M_va, y_va, class_weight=None if best_cfg["class_weight"] is None else balanced_w, seed=seed, verbose=False)
    seed_rows.append({"seed": seed, **val_scores(predict_proba_seq(net, Xi_va, M_va)), "best_epoch": h["best_epoch"]})
seed_tbl = pd.DataFrame(seed_rows).set_index("seed")
display(seed_tbl)
print("mean ± std：", seed_tbl[["accuracy", "macro_f1"]].agg(["mean", "std"]).round(4).to_dict())
best_cfg["epochs"] = int(round(seed_tbl["best_epoch"].mean()))

In [ ]:
# TODO 11（問答）調參
#
# (1) 5A：
#     (a) H 從 32 到 256，train accuracy 和 val accuracy 各怎麼變？兩條線的差距（過擬合）在哪個 H 開始變大？
#     (b) 參數量變幾倍？val 成績值得嗎？用 one-standard-error rule（差距小於 5C 的 std 就選簡單的）你會選哪個 H？
# 答：
#
# (2) 5B：
#     (a) class weight = balanced 讓 accuracy、macro-F1 各怎麼變？為什麼會「一升一降」或「兩個都降」？
#         （提示：權重讓小類別的錯誤「比較貴」→ 模型更常猜小類別 → 大類別被誤判成小類別的次數？）
#     (b) Bi-RNN 的進步，在有／沒有 class weight 時一樣嗎？
# 答：
#
# (3) 5C：3 個 seed 的 macro-F1 標準差是多少？5B 裡各設定之間的差距，有比這個雜訊大嗎？
# 答：

---
## 6. 測試集評估

依 README：用最佳設定（5B／5C）在**完整訓練集**（train + validation）refit，epoch 數用 5C 的 best epoch 平均，測試集只評估一次。

其他模型（字母 n-gram + LogReg、手寫 NumPy RNN、PyTorch 預設 RNN）用的是只在 train 上訓練的版本，當參照點；n-gram + LogReg 另外也在 train + val 上 refit 一次（公平比較）。

In [ ]:
from learning_map.metrics import classification_metrics
from learning_map.results import compare, log_result

BENCH = "names_lang"
Xi_full = np.vstack([np.pad(Xi_tr, ((0, 0), (0, max(0, Xi_va.shape[1] - Xi_tr.shape[1])))), np.pad(Xi_va, ((0, 0), (0, max(0, Xi_tr.shape[1] - Xi_va.shape[1]))))])
M_full = (Xi_full > 0).astype(float)
y_full = np.concatenate([y_tr, y_va])
print("最終設定：", best_cfg)

set_seed(0)
net_final = CharRNNTorch(V, K, bidirectional=best_cfg["bidirectional"])
train_seq(net_final, Xi_full, M_full, y_full, n_epochs=best_cfg["epochs"],
          class_weight=None if best_cfg["class_weight"] is None else len(y_full) / (K * np.bincount(y_full, minlength=K)), verbose=False)
ngram_full = make_pipeline(CountVectorizer(analyzer="char_wb", ngram_range=(1, 3)), LogisticRegression(max_iter=3000)).fit(
    pd.concat([df_tr["ascii"], df_va["ascii"]]), y_full)

# 名稱: (測試集機率 (n_test, K), 要記下的參數)
eval_models = {
    "char 1-3-gram + LogReg (train+val)": (ngram_full.predict_proba(df_te["ascii"]), {"ngram": [1, 3]}),
    "RNN (scratch NumPy, SGD)": (rnn_np.predict_proba(Xi_te, M_te), {"H": 128, "lr": rnn_np.lr, "epochs": rnn_np.best_epoch_}),
    "RNN (PyTorch, Adam)": (predict_proba_seq(net_rnn, Xi_te, M_te), {"H": 128, "epochs": hist_rnn["best_epoch"]}),
    "RNN tuned (PyTorch, train+val)": (predict_proba_seq(net_final, Xi_te, M_te), {"H": 128, **best_cfg}),
}
test_scores = {}
for name, (prob, params) in eval_models.items():
    m = classification_metrics(y_te, prob.argmax(axis=1), average="macro")
    m["log_loss"] = round(log_loss(y_te, np.clip(prob, 1e-12, 1), labels=np.arange(K)), 4)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "f1")

---
## 7. 診斷

1. **混淆矩陣**（每一列除以該類的數量 = 召回率）：哪些語言互相混淆？
2. **每一類的 F1 vs 訓練資料量**：小類別一定比較差嗎？RNN 和 n-gram + LogReg 各在哪些類別贏？
3. **「最有把握地錯」的名字**：是模型的問題，還是名字本身就模糊（例如出現在兩種語言）？

In [ ]:
from learning_map.metrics import plot_confusion_matrix

prob_final = predict_proba_seq(net_final, Xi_te, M_te)
pred_final = prob_final.argmax(axis=1)
fig, ax = plt.subplots(figsize=(11, 9))
plot_confusion_matrix(y_te, pred_final, labels=LANGS, normalize="true", ax=ax)
for t in ax.texts:
    t.set_fontsize(7)
plt.show()

prob_ng = ngram_full.predict_proba(df_te["ascii"])
per_class = pd.DataFrame({
    "train count": counts_tr,
    "F1 RNN tuned": f1_score(y_te, pred_final, average=None, labels=np.arange(K), zero_division=0),
    "F1 n-gram LogReg": f1_score(y_te, prob_ng.argmax(axis=1), average=None, labels=np.arange(K), zero_division=0),
}, index=LANGS).round(3)
per_class["RNN − n-gram"] = per_class["F1 RNN tuned"] - per_class["F1 n-gram LogReg"]
display(per_class.sort_values("train count", ascending=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(per_class["train count"], per_class["F1 RNN tuned"], label="RNN tuned")
ax.scatter(per_class["train count"], per_class["F1 n-gram LogReg"], marker="x", label="n-gram LogReg")
for lang, r in per_class.iterrows():
    ax.annotate(lang, (r["train count"], r["F1 RNN tuned"]), fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("training examples (log)")
ax.set_ylabel("test F1")
ax.legend()
plt.show()

In [ ]:
# 最有把握地錯的 15 個名字；以及「出現在多種語言」的名字的表現
conf_wrong = np.flatnonzero(pred_final != y_te)
conf_wrong = conf_wrong[np.argsort(-prob_final[conf_wrong].max(axis=1))][:15]
all_langs_of = df.groupby("ascii")["language"].agg(lambda s: sorted(set(s)))
display(pd.DataFrame({
    "name": df_te["ascii"].iloc[conf_wrong].to_numpy(),
    "true": [LANGS[k] for k in y_te[conf_wrong]],
    "pred": [LANGS[k] for k in pred_final[conf_wrong]],
    "p(pred)": prob_final[conf_wrong].max(axis=1).round(3),
    "all labels of this name": [all_langs_of[nm] for nm in df_te["ascii"].iloc[conf_wrong]],
}))
ambiguous = df_te["ascii"].map(lambda nm: len(all_langs_of[nm]) > 1).to_numpy()
print(f"test 裡屬於多種語言的名字：{ambiguous.sum()} 筆；accuracy {np.mean(pred_final[ambiguous] == y_te[ambiguous]):.3f}"
      f"（其他名字 {np.mean(pred_final[~ambiguous] == y_te[~ambiguous]):.3f}）")

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 測試集：
#     (a) 依 macro-F1 排序：n-gram + LogReg、手寫 RNN、PyTorch RNN、調過的 RNN。跟 validation 的排名一樣嗎？跟你在 TODO 2 (3) 的預測一樣嗎？
#     (b) 如果 RNN 沒有贏過 n-gram + LogReg：這個資料集的哪些性質讓「3 個字母的局部順序」就夠用？（提示：名字的長度、字尾、資料量）
#         什麼樣的任務 RNN 才有明顯的優勢？（提示：需要記住很遠的資訊、或輸出也是序列）
# 答：
#
# (2) 混淆矩陣：最常混淆的三組語言是哪些？用語言學解釋（例如同一個語系、同一套拼寫規則、歷史上的移民）。
# 答：
#
# (3) 每一類的 F1：小類別一定比較差嗎？有沒有「資料很少、F1 卻很高」的語言？為什麼？（提示：字尾表）
# 答：
#
# (4) 最有把握地錯：有幾個名字其實「也屬於」被預測的那個語言？屬於多種語言的名字，accuracy 是多少？
# 答：
#
# (5) RNN 的極限（為下一課 LSTM 鋪路）：
#     (a) 實驗 1 裡，梯度傳到 11 步之前剩多少？如果名字有 50 個字母，第 1 個字母能影響參數的更新嗎？
#     (b) LSTM 的「閘門」要解決的就是這件事。你猜它會讓哪一條曲線變平？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

名字分類（18 種語言；去重、依名字分組的分層切分）

| 模型 | 做法 | 關鍵性質 | val acc | val macro-F1 | test acc | test macro-F1 |
| --- | --- | --- | --- | --- | --- | --- |
| Dummy prior | | | | | — | — |
| char 1～3-gram + LogReg | | | | | | |
| RNN（手寫 NumPy，SGD） | | | | | | |
| RNN（PyTorch，Adam） | | | | | | |
| Bi-RNN（PyTorch） | | | | | — | — |
| RNN tuned | | | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ BPTT 的完整推導與截斷 BPTT（truncated BPTT）
- ☐ 梯度消失／爆炸的分析（Bengio, Simard & Frasconi 1994；Pascanu, Mikolov & Bengio 2013）
- ☐ Adam 最佳化器（動量 + 每個參數自己的步長）
- ☐ 正交初始化、identity 初始化（IRNN）

**延伸**
- ☐ 多層（stacked）RNN、dropout 用在 RNN
- ☐ 下一課 LSTM／GRU：閘門如何讓梯度「走捷徑」
- ☐ 字元級語言模型（生成名字，原始教學的第三部分「Generating Names」）
- ☐ Seq2Seq（第四部分）：輸入和輸出都是序列